In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

/kaggle/input/competitions/heart-disease-prediction-shohoj-coding-mlb-2/sample_submission.csv
/kaggle/input/competitions/heart-disease-prediction-shohoj-coding-mlb-2/train.csv
/kaggle/input/competitions/heart-disease-prediction-shohoj-coding-mlb-2/test.csv


In [2]:
train= pd.read_csv('/kaggle/input/competitions/heart-disease-prediction-shohoj-coding-mlb-2/train.csv')
test= pd.read_csv('/kaggle/input/competitions/heart-disease-prediction-shohoj-coding-mlb-2/test.csv')
print(train.tail())
print(test.head())

      id   age  anaemia  creatinine_phosphokinase  diabetes  \
234  229  72.0        0                       211         0   
235  131  60.0        1                      1082         1   
236  177  49.0        1                        69         0   
237  112  50.0        0                       369         1   
238   28  58.0        1                        60         0   

     ejection_fraction  high_blood_pressure  platelets  serum_creatinine  \
234                 25                    0   274000.0               1.2   
235                 45                    0   250000.0               6.1   
236                 50                    0   132000.0               1.0   
237                 25                    0   252000.0               1.6   
238                 38                    0   153000.0               5.8   

     serum_sodium  sex  smoking  time  DEATH_EVENT  
234           134    0        0   207            0  
235           131    1        0   107            0  
236  

In [3]:
test_id = test['id'] if 'id' in test.columns else test.index

In [4]:
def create_features(df):
    data = df.copy()
    data['creatinine_ef_ratio'] = data['serum_creatinine'] / (data['ejection_fraction'] + 1e-5)
    data['high_risk_creatinine'] = (data['serum_creatinine'] > 1.5).astype(int)
    data['low_ejection_fraction'] = (data['ejection_fraction'] < 30).astype(int)
    
    data['age_creatinine_interaction'] = data['age'] * data['serum_creatinine']
    
    data['total_risk_factors'] = data['anaemia'] + data['diabetes'] + data['high_blood_pressure'] + data['smoking']
    
    return data

train = create_features(train)
test = create_features(test)

features = [col for col in train.columns if col not in ['id', 'DEATH_EVENT']]
X = train[features]
y = train['DEATH_EVENT']
X_test = test[features]

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.02,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# 5. Local Validation Check
val_preds = model.predict(X_val)
acc = accuracy_score(y_val, val_preds)
print(f"Local Validation Accuracy: {acc * 100:.2f}%")

Local Validation Accuracy: 81.25%


In [7]:
model.fit(X, y)
final_predictions = model.predict(X_test)

submission = pd.DataFrame({
    'id': test_id,
    'DEATH_EVENT': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("Success! 'submission.csv' has been generated and is ready to submit.")

Success! 'submission.csv' has been generated and is ready to submit.
